In [ ]:
# %pip install isotree
# %pip install pyod
# %pip install combo

In [ ]:
# Librerie
import pandas as pd
import numpy as np

# High Dimensionality
from pyod.models.abod import ABOD

# Ensemble
from pyod.models.feature_bagging import FeatureBagging
# carico LOF per FeaBag, chiaramente si può cambiare a seconda del modello semplice scelto
from pyod.models.lof import LOF
from pyod.models.loda import LODA

# Foreste
from sklearn.ensemble import IsolationForest
from isotree import IsolationForest as EIF

In [ ]:
# Dati
train = pd.read_csv("train_imputed.csv")
test = pd.read_csv("test_imputed.csv")

In [ ]:
####
# High Dimensionality
####

In [ ]:
# ABOD

#!!! LENTO, su portatile in carica ci mette 5+ minuti; sul fisso un paio

# Contamination è la proporzione di outliers; n_neighbors autoesplicativo
abod_model = ABOD(contamination=0.01, n_neighbors=76)
abod_model.fit(train)

# 0 = inliers e 1 = outliers
abod_preds = abod_model.predict(train)
abod_scores = abod_model.decision_scores_

In [ ]:
####
# Ensemble
####

In [ ]:
# FeaBag

feabag_model = FeatureBagging(
    base_estimator=LOF(),  # stimatore base, si può cambiare
    n_estimators=100,  # da testare coi dati veri per capire quanto ci vuole
    max_features=0.5,  # proporzione di variabili da estrarre per ogni modello
    contamination=0.01,  # come prima, 1% di outliers
    random_state=222
)
feabag_model.fit(train)

feabag_preds = feabag_model.predict(train)

In [ ]:
# LODA

# n_bins si lascia ottimizzare al modello; n_random_cuts è praticamente il numero di vettori generati
loda_model = LODA(contamination=0.01, n_bins='auto', n_random_cuts=100)
loda_model.fit(train)

loda_preds = loda_model.predict(train)

In [ ]:
####
# Foreste
####

In [ ]:
# Random Forest

isolation_forest_model = IsolationForest(
    n_estimators=100,  # da testare coi dati veri per capire quanto ci vuole, NUMERO di alberi
    # dimensione del campione per ogni albero, sembra che 256 abbia base scientifica
    max_samples=256,
    contamination=0.01,  # usual
    random_state=222
)
isolation_forest_preds = isolation_forest_model.fit_predict(train)

In [ ]:
# Extended Forest


# In isotree, ndim=1 equivale a Isolation Forest standard.
# Scegliere ndim=df.shape[1] per la Extended Isolation Forest completa.
extended_forest_model = EIF(
    sample_size=256,  # come prima
    ntrees=100,  # numero alberi
    # serve a dire al modello che vogliamo la versione estesa
    ndim=train.shape[1],
    random_seed=222
)
extended_forest_model.fit(train)

# EIF genera uno score (più alto = più probabile outlier)
extended_forest_scores = extended_forest_model.predict(train)

# gli outlier vanno trovati a mano
threshold = np.percentile(extended_forest_scores, 99)  # 99 percentile
extended_forest_preds = (extended_forest_scores > threshold).astype(int)